In [1]:
import boto3

In [3]:
session = boto3.Session(
    region_name='us-east-1',
    aws_access_key_id='test',
    aws_secret_access_key='test'
)

endpoint = 'http://localhost:4566'

In [4]:
kinesis = session.client('kinesis', endpoint_url=endpoint)

In [17]:
shards = kinesis.list_shards(StreamName='pr-events-stream')

In [22]:
shards['Shards']

[{'ShardId': 'shardId-000000000000',
  'HashKeyRange': {'StartingHashKey': '0',
   'EndingHashKey': '340282366920938463463374607431768211455'},
  'SequenceNumberRange': {'StartingSequenceNumber': '0',
   'EndingSequenceNumber': '1795296782364'}},
 {'ShardId': 'shardId-000000000001',
  'ParentShardId': 'shardId-000000000000',
  'HashKeyRange': {'StartingHashKey': '0',
   'EndingHashKey': '170141183460469231731687303715884105727'},
  'SequenceNumberRange': {'StartingSequenceNumber': '1795296782364'}},
 {'ShardId': 'shardId-000000000002',
  'ParentShardId': 'shardId-000000000000',
  'HashKeyRange': {'StartingHashKey': '170141183460469231731687303715884105728',
   'EndingHashKey': '340282366920938463463374607431768211455'},
  'SequenceNumberRange': {'StartingSequenceNumber': '1795296782364'}}]

In [39]:
last_sequence_number = {}
for shard in shards['Shards']:
    shard_iterator = kinesis.get_shard_iterator(StreamName='pr-events-stream', ShardId=shard['ShardId'],ShardIteratorType='TRIM_HORIZON')
    records = kinesis.get_records(ShardIterator=shard_iterator['ShardIterator'], Limit=25)
    record_details = records.get('Records')
    for i in record_details:
        print(i['SequenceNumber'], i['PartitionKey'])
    if len(record_details) > 0:
        last_sequence_number[shard['ShardId']] = i['SequenceNumber']

1795296782367 org/repo-b
1795296782368 org/repo-c
1795296782371 human
1795296782374 human
1795296782376 human
1795296782365 org/repo-a
1795296782366 org/repo-a
1795296782369 org/repo-a
1795296782370 org/repo-d
1795296782372 agent
1795296782373 agent
1795296782375 agent


In [40]:
last_sequence_number

{'shardId-000000000001': '1795296782376',
 'shardId-000000000002': '1795296782375'}

In [42]:
records

{'Records': [{'SequenceNumber': '1795296782365',
   'ApproximateArrivalTimestamp': datetime.datetime(2026, 11, 21, 21, 2, 13, 12000, tzinfo=tzlocal()),
   'Data': b'{"repo": "org/repo-a", "pr_number": 101, "author_type": "human", "ci_skipped": false, "ingested_at": 1795312932.996715}',
   'PartitionKey': 'org/repo-a'},
  {'SequenceNumber': '1795296782366',
   'ApproximateArrivalTimestamp': datetime.datetime(2026, 11, 21, 21, 2, 13, 30000, tzinfo=tzlocal()),
   'Data': b'{"repo": "org/repo-a", "pr_number": 102, "author_type": "agent", "ci_skipped": true, "ingested_at": 1795312933.0286999}',
   'PartitionKey': 'org/repo-a'},
  {'SequenceNumber': '1795296782369',
   'ApproximateArrivalTimestamp': datetime.datetime(2026, 11, 21, 21, 2, 13, 39000, tzinfo=tzlocal()),
   'Data': b'{"repo": "org/repo-a", "pr_number": 103, "author_type": "agent", "ci_skipped": false, "ingested_at": 1795312933.038869}',
   'PartitionKey': 'org/repo-a'},
  {'SequenceNumber': '1795296782370',
   'ApproximateArriva

In [73]:
latest_iterator = {}
for shard in shards['Shards']:
    response = kinesis.get_shard_iterator(
        StreamName='pr-events-stream', 
        ShardId=shard['ShardId'],
        ShardIteratorType ='TRIM_HORIZON'
    )
    iterator = response['ShardIterator']
    latest_iterator[shard['ShardId']] = iterator

In [74]:
for shard in shards['Shards']:
    response = kinesis.get_records(ShardIterator=latest_iterator[shard['ShardId']])
    print(f'{shard['ShardId']}: {len(response['Records'])}')
    latest_iterator[shard['ShardId']] = response['NextShardIterator']
    for record in response['Records']:
        print(record['SequenceNumber'])

shardId-000000000000: 0
shardId-000000000001: 5
1795296782367
1795296782368
1795296782371
1795296782374
1795296782376
shardId-000000000002: 17
1795296782365
1795296782366
1795296782369
1795296782370
1795296782372
1795296782373
1795296782375
1795296782377
1795296782378
1795296782379
1795296782380
1795296782381
1795296782382
1795296782383
1795296782384
1795296782385
1795296782386


In [68]:
response

{'Records': [],
 'NextShardIterator': 'cHItZXZlbnRzLXN0cmVhbXxzaGFyZElkLTAwMDAwMDAwMDAwMnxBRlRFUl9TRVFVRU5DRV9OVU1CRVJ8MTc5NTI5Njc4MjM4NnwwfA==',
 'MillisBehindLatest': 0,
 'ResponseMetadata': {'RequestId': 'f27ce2eb-9bde-4387-9dc4-a7dfb9b222ef',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'content-type': 'application/x-amz-json-1.1;charset=UTF-8',
   'content-length': '164',
   'date': 'Sun, 22 Nov 2026 21:26:26 GMT',
   'x-amz-id-2': 'f27ce2eb-9bde-4387-9dc4-a7dfb9b222ef',
   'x-amz-request-id': 'f27ce2eb-9bde-4387-9dc4-a7dfb9b222ef',
   'x-amzn-requestid': 'f27ce2eb-9bde-4387-9dc4-a7dfb9b222ef'},
  'RetryAttempts': 0}}

In [99]:
latest_iterator = {}
for shard in shards['Shards']:
    response = kinesis.get_shard_iterator(
        StreamName='pr-events-stream', 
        ShardId=shard['ShardId'],
        ShardIteratorType ='AFTER_SEQUENCE_NUMBER',
        StartingSequenceNumber='1795296782372'
    )
    iterator = response['ShardIterator']
    latest_iterator[shard['ShardId']] = iterator

latest_iterator

{'shardId-000000000000': 'cHItZXZlbnRzLXN0cmVhbXxzaGFyZElkLTAwMDAwMDAwMDAwMHxBRlRFUl9TRVFVRU5DRV9OVU1CRVJ8MTc5NTI5Njc4MjM3MnwwfA==',
 'shardId-000000000001': 'cHItZXZlbnRzLXN0cmVhbXxzaGFyZElkLTAwMDAwMDAwMDAwMXxBRlRFUl9TRVFVRU5DRV9OVU1CRVJ8MTc5NTI5Njc4MjM3MnwwfA==',
 'shardId-000000000002': 'cHItZXZlbnRzLXN0cmVhbXxzaGFyZElkLTAwMDAwMDAwMDAwMnxBRlRFUl9TRVFVRU5DRV9OVU1CRVJ8MTc5NTI5Njc4MjM3MnwwfA=='}

In [93]:
response = kinesis.get_records(ShardIterator=latest_iterator[shard['ShardId']])

In [100]:
response = kinesis.get_shard_iterator(
    StreamName='pr-events-stream', 
    ShardId='shardId-000000000002',
    ShardIteratorType ='AFTER_SEQUENCE_NUMBER',
    StartingSequenceNumber='1795296782372'
)

In [101]:
check = kinesis.get_records(ShardIterator=response['ShardIterator'])
len(check['Records'])

12

In [102]:
# simulate fail and request iterator with IteratorType: AFTER_SEQUENCE_NUMBER

In [103]:
class SimulateError(Exception):
    pass

In [111]:
latest_iterator = {}
for shard in shards['Shards']:
    shard_id = shard['ShardId']
    response = kinesis.get_shard_iterator(
        StreamName='pr-events-stream',
        ShardIteratorType='TRIM_HORIZON',
        ShardId=shard_id
    )
    latest_iterator[shard_id] = response['ShardIterator']

In [120]:
last_sequence_number = {}

for shard in shards['Shards']:
    shard_id = shard['ShardId']
    response = kinesis.get_records(ShardIterator=latest_iterator[shard_id])
    records = response['Records']
    if len(records) > 0:
        try:
            for i in range(2):
                print(f'ShardId: {shard_id}, SequenceNumber: {records[i]['SequenceNumber']}')
            raise SimulateError('EEEEEEE try to request new ShardIterator')
        except SimulateError as e:
            last_sequence_number[shard_id] = records[i]['SequenceNumber']

ShardId: shardId-000000000001, SequenceNumber: 1795296782367
ShardId: shardId-000000000001, SequenceNumber: 1795296782368
ShardId: shardId-000000000002, SequenceNumber: 1795296782365
ShardId: shardId-000000000002, SequenceNumber: 1795296782366


In [125]:
resume_iterator = {}
for shard in shards['Shards'][1:]:
    shard_id = shard['ShardId']
    response = kinesis.get_shard_iterator(
        StreamName = 'pr-events-stream',
        ShardId = shard_id,
        ShardIteratorType = 'AFTER_SEQUENCE_NUMBER',
        StartingSequenceNumber = last_sequence_number[shard_id]
    )
    resume_iterator[shard_id] = response['ShardIterator']

In [130]:
for shard in shards['Shards'][1:]:
    shard_id = shard['ShardId']
    response = kinesis.get_records(ShardIterator=resume_iterator[shard_id])
    print(f'ShardId: {shard_id} - {len(response['Records'])}')

ShardId: shardId-000000000001 - 3
ShardId: shardId-000000000002 - 15


In [131]:
# save data into s3 bucket

In [132]:
s3 = session.client('s3', endpoint_url=endpoint)

In [141]:
import json
from datetime import date, datetime


def json_serial(obj):
  """JSON serializer for objects not serializable by default json code"""
  if isinstance(obj, (datetime, date)):
    return obj.isoformat()
  if isinstance(obj, bytes):
      return obj.decode("utf-8")
  raise TypeError(f"Type {type(obj)} not serializable")


for shard in shards['Shards']:
    shard_id = shard['ShardId']
    response = kinesis.get_shard_iterator(
        StreamName='pr-events-stream',
        ShardId=shard_id,
        ShardIteratorType='TRIM_HORIZON',
    )
    iterator = response['ShardIterator']
    get_records = kinesis.get_records(ShardIterator=iterator)
    records = get_records['Records']
    if len(records) > 0:
        for record in records:
            s3.put_object(
                Bucket='data-platform-raw',
                Key=f'streaming/pr-events/{shard_id}-{record['SequenceNumber']}.json',
                Body=json.dumps(record, indent=2, default=json_serial),
                ContentType="application/json"
            )